# scOPE end-to-end -- SKCM (TCGA Skin Cutaneous Melanoma)

Two-phase scOPE workflow applied to melanoma:

1. **Phase 1** -- learn a latent space from TCGA SKCM bulk RNA-seq (~470
   samples) and train mutation classifiers (BRAF, NRAS, NF1, TP53, ...).
2. **Phase 2** -- project Jerby-Arnon et al. 2018 melanoma scRNA-seq
   (GSE115978, 7 186 cells, 31 tumours) into the bulk latent space and infer
   per-cell mutation probabilities.

**Data notes:**
- Bulk expression: UCSC Xena TCGA SKCM `HiSeqV2_PANCAN`
  (log2(norm_count + 1), genes x samples).  Both primary (01) and metastatic
  (06) samples are retained.
- Mutations: TCGA MC3 PanCanAtlas MAF (shared with CRC/BRCA notebooks).
  Filtered to SKCM samples by matching against bulk obs_names.
- scRNA-seq: Jerby-Arnon et al. 2018 (GSE115978) -- TPM matrix (log2(TPM+1))
  downloaded via GEO HTTPS endpoint. No further normalisation applied.

**BRAF V600E note:**
BRAF V600E is a hotspot detectable at the RNA level (allele-specific
expression). After projection, you can optionally validate scOPE's predicted
BRAF probability against SComatic or RESA calls from the scRNA-seq BAMs.


## 1. Imports & paths

In [1]:
import os

import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import matplotlib.pyplot as plt
import requests
from sklearn.isotonic import IsotonicRegression

from scope import BulkPipeline, SingleCellPipeline
from scope.visualization import (
    compute_umap,
    plot_mutation_probabilities,
    plot_scree,
    plot_mutation_heatmap,
)


In [2]:
# -- Paths ------------------------------------------------------------------
BASE_DIR   = "/Users/ashforda/Desktop/Pathways + Omics/scOPE/scOPE_project_overhaul/scOPE/data"
BULK_DIR   = os.path.join(BASE_DIR, "TCGA_SKCM")
SC_DIR     = os.path.join(BASE_DIR, "SKCM_scRNA")
MODELS_DIR = os.path.join(BASE_DIR, "..", "models", "SKCM")

for d in [BULK_DIR, SC_DIR, MODELS_DIR]:
    os.makedirs(d, exist_ok=True)

# -- File locations ---------------------------------------------------------
skcm_expr_path = os.path.join(BULK_DIR, "TCGA_SKCM_HiSeqV2_PANCAN.tsv.gz")
mc3_path       = os.path.join(BASE_DIR, "mc3.v0.2.8.PUBLIC.xena.gz")
sc_tpm_path    = os.path.join(SC_DIR,   "GSE115978_tpm.csv.gz")
# GEO uses dot separator: GSE115978_cell.annotations.csv.gz
sc_meta_path   = os.path.join(SC_DIR,   "GSE115978_cell.annotations.csv.gz")


In [3]:
import scope

print(f'scOPE v{scope.__version__}')


scOPE v0.1.6


## 2. Download raw data

In [4]:
# -- UCSC Xena -- bulk RNA-seq (expression files still work) ---------------
XENA_BASE = "https://tcga.xenahubs.net/download"

if os.path.exists(skcm_expr_path):
    print(f"  already present -- {os.path.basename(skcm_expr_path)}")
else:
    print(f"  downloading {os.path.basename(skcm_expr_path)} ...")
    r = requests.get(f"{XENA_BASE}/TCGA.SKCM.sampleMap/HiSeqV2_PANCAN.gz",
                     stream=True, timeout=300)
    r.raise_for_status()
    with open(skcm_expr_path, "wb") as fh:
        for chunk in r.iter_content(1 << 20):
            fh.write(chunk)
    print(f"  done -> {skcm_expr_path}")


  already present -- TCGA_SKCM_HiSeqV2_PANCAN.tsv.gz


In [5]:
# -- MC3 PanCanAtlas MAF -- all TCGA cancer types, one file (~200 MB) -------
# Shared resource -- check CRC and BRCA directories before re-downloading.
for _alt in [
    os.path.join(BASE_DIR, "TCGA_CRC",  "mc3.v0.2.8.PUBLIC.xena.gz"),
    os.path.join(BASE_DIR, "TCGA_BRCA", "mc3.v0.2.8.PUBLIC.xena.gz"),
]:
    if not os.path.exists(mc3_path) and os.path.exists(_alt):
        import shutil
        shutil.copy(_alt, mc3_path)
        print(f"  copied MC3 from {os.path.dirname(_alt)} -> {mc3_path}")
        break

if not os.path.exists(mc3_path):
    print("Downloading MC3 MAF (~200 MB) ...")
    r = requests.get(
        "https://pancanatlas.xenahubs.net/download/mc3.v0.2.8.PUBLIC.xena.gz",
        stream=True, timeout=600,
    )
    r.raise_for_status()
    with open(mc3_path, "wb") as fh:
        for chunk in r.iter_content(1 << 20):
            fh.write(chunk)
    print(f"  done -> {mc3_path}")
else:
    print(f"  already present -- {os.path.basename(mc3_path)}")


  already present -- mc3.v0.2.8.PUBLIC.xena.gz


In [6]:
# -- GEO -- Jerby-Arnon et al. 2018 GSE115978 ------------------------------
# TPM matrix  : genes x cells, log2(TPM+1)
# Cell metadata: cell type, tumour, malignancy label
# Note: GEO filename is GSE115978_cell.annotations.csv.gz (dot separator, csv)
GEO_HTTP = "https://www.ncbi.nlm.nih.gov/geo/download"

sc_downloads = {
    sc_tpm_path  : (f"{GEO_HTTP}/?acc=GSE115978&format=file"
                    "&file=GSE115978%5Ftpm%2Ecsv%2Egz"),
    sc_meta_path : (f"{GEO_HTTP}/?acc=GSE115978&format=file"
                    "&file=GSE115978%5Fcell%2Eannotations%2Ecsv%2Egz"),
}
for dest, url in sc_downloads.items():
    if os.path.exists(dest):
        print(f"  already present -- {os.path.basename(dest)}")
    else:
        print(f"  downloading {os.path.basename(dest)} ...")
        r = requests.get(url, stream=True, timeout=600)
        r.raise_for_status()
        with open(dest, "wb") as fh:
            for chunk in r.iter_content(1 << 20):
                fh.write(chunk)
        print(f"  done -> {dest}")


  already present -- GSE115978_tpm.csv.gz
  already present -- GSE115978_cell.annotations.csv.gz


## 3. Load & prepare bulk RNA-seq

SKCM includes both primary (01) and metastatic (06) tumours; both are retained
since the scRNA-seq cohort is also mixed.


In [7]:
# -- Peek -------------------------------------------------------------------
peek = pd.read_csv(skcm_expr_path, sep="\t", index_col=0, nrows=3)
print(f"Peek : {peek.shape}   cols[:4] = {peek.columns[:4].tolist()}")


Peek : (3, 474)   cols[:4] = ['TCGA-YD-A89C-06', 'TCGA-Z2-AA3V-06', 'TCGA-EB-A3Y6-01', 'TCGA-EE-A3JA-06']


In [8]:
df_bulk = pd.read_csv(skcm_expr_path, sep="\t", index_col=0).T   # samples x genes

# TCGA sample type codes:
#   01 = primary tumour, 06 = metastatic, 11 = normal solid tissue
# Including normals gives the classifier true negatives with distinct
# transcriptional profiles -- this pulls the TME probability floor down.
SAMPLE_TYPE_MAP = {"01": "primary", "06": "metastatic", "11": "normal"}
df_bulk = df_bulk[df_bulk.index.str[13:15].isin(SAMPLE_TYPE_MAP)]

adata_bulk = ad.AnnData(
    X   = df_bulk.values.astype(np.float32),
    obs = pd.DataFrame(
            {"sample_type": df_bulk.index.str[13:15].map(SAMPLE_TYPE_MAP)},
            index=df_bulk.index,
          ),
    var = pd.DataFrame(index=df_bulk.columns),
)
adata_bulk.var_names_make_unique()
print(f"Bulk loaded : {adata_bulk.n_obs} samples x {adata_bulk.n_vars} genes")
print(adata_bulk.obs["sample_type"].value_counts())


Bulk loaded : 473 samples x 20530 genes
sample_type
metastatic    368
primary       104
normal          1
Name: count, dtype: int64


In [9]:
# -- Sanity check -----------------------------------------------------------
X = adata_bulk.X
print(f"NaN : {np.isnan(X).sum()} | Inf : {np.isinf(X).sum()}")
print(f"Min : {np.nanmin(X):.3f} | Max : {np.nanmax(X):.3f}")
print(f"Neg : {(X < 0).sum()}  (expected for log2 data)")


NaN : 0 | Inf : 0
Min : -12.348 | Max : 19.377
Neg : 5708241  (expected for log2 data)


## 4. Build mutation label matrix

SKCM has very high TMB (UV signature). We focus on major driver genes where
expression-linked signatures are strong (BRAF, NRAS, NF1, TP53, CDKN2A).
SKCM samples are identified by matching against `adata_bulk.obs_names`.


In [10]:
# -- Key SKCM driver genes --------------------------------------------------
SKCM_GENES = [
    # -- MAPK pathway -------------------------------------------------------
    "BRAF",     # ~50% -- V600E hotspot; defines targeted-therapy eligibility
    "NRAS",     # ~30% -- Q61 hotspot; mutually exclusive with BRAF
    "NF1",      # ~15% -- RASopathy gene; third major melanoma driver
    "MAP2K1",   # MEK1
    "MAP2K2",   # MEK2
    "KRAS",
    "HRAS",

    # -- PI3K/AKT pathway ---------------------------------------------------
    "PTEN",     # loss common in BRAF-mutant tumours
    "PIK3CA",
    "AKT1",
    "AKT3",

    # -- Tumour suppressors -------------------------------------------------
    "TP53",
    "CDKN2A",   # p16/ARF locus -- frequent deletion

    # -- WNT / differentiation ----------------------------------------------
    "CTNNB1",
    "APC",

    # -- Chromatin / epigenetic ---------------------------------------------
    "ARID2",
    "SETD2",
    "KDM6A",
    "ARID1A",

    # -- Other recurrent ----------------------------------------------------
    "RAC1",     # P29S -- UV hotspot
    "PPP6C",    # UV hotspot -- MAPK pathway regulation
    "PREX2",    # PI3K pathway
    "IDH1",
    "KIT",
    "GNAQ",     # uveal melanoma
    "GNA11",    # uveal melanoma
]
SKCM_GENES = list(dict.fromkeys(SKCM_GENES))
print(f"Targeting {len(SKCM_GENES)} SKCM driver genes")


Targeting 26 SKCM driver genes


In [11]:
KEEP_CLASSES = {
    "Missense_Mutation", "Nonsense_Mutation", "Frame_Shift_Del",
    "Frame_Shift_Ins", "In_Frame_Del", "In_Frame_Ins",
    "Splice_Site", "Translation_Start_Site", "Nonstop_Mutation",
}


In [12]:
# -- Load MC3, filter to SKCM by matching bulk obs_names -------------------
mc3 = pd.read_csv(mc3_path, sep="\t", low_memory=False)
print(f"MC3 raw : {len(mc3):,} rows   cols[:6]: {mc3.columns.tolist()[:6]}")

mc3 = mc3.rename(columns={
    "sample" : "Tumor_Sample_Barcode",
    "effect" : "Variant_Classification",
    "gene"   : "Hugo_Symbol",
})

# Filter to SKCM by matching against bulk 15-char barcodes
bulk_15_set = set(adata_bulk.obs_names.str[:15])
mc3["sample_id"] = mc3["Tumor_Sample_Barcode"].str[:15]
maf_skcm = mc3[mc3["sample_id"].isin(bulk_15_set)].copy()
print(f"SKCM rows (pre-filter) : {len(maf_skcm):,}   samples: {maf_skcm['sample_id'].nunique()}")

maf_skcm = maf_skcm[maf_skcm["Variant_Classification"].isin(KEEP_CLASSES)]
maf_all  = maf_skcm[["sample_id", "Hugo_Symbol"]].dropna()
print(f"After coding filter    : {len(maf_all):,} variants   {maf_all['sample_id'].nunique()} samples")


MC3 raw : 2,907,335 rows   cols[:6]: ['sample', 'chr', 'start', 'end', 'reference', 'alt']
SKCM rows (pre-filter) : 412,743   samples: 465
After coding filter    : 242,965 variants   465 samples


In [13]:
# -- Build binary mutation matrix (samples x genes) ------------------------
mut_matrix = (
    maf_all[["sample_id", "Hugo_Symbol"]]
    .drop_duplicates()
    .assign(mutated=1)
    .pivot_table(index="sample_id", columns="Hugo_Symbol",
                 values="mutated", fill_value=0)
)
mut_matrix.columns.name = None
mut_matrix.index.name   = None

genes_present = [g for g in SKCM_GENES if g in mut_matrix.columns]
genes_missing = [g for g in SKCM_GENES if g not in mut_matrix.columns]
mutation_labels = mut_matrix[genes_present]

print(f"Mutation matrix : {mutation_labels.shape}")
print(f"Genes found     : {genes_present}")
if genes_missing:
    print(f"Genes missing   : {genes_missing}")


Mutation matrix : (465, 26)
Genes found     : ['BRAF', 'NRAS', 'NF1', 'MAP2K1', 'MAP2K2', 'KRAS', 'HRAS', 'PTEN', 'PIK3CA', 'AKT1', 'AKT3', 'TP53', 'CDKN2A', 'CTNNB1', 'APC', 'ARID2', 'SETD2', 'KDM6A', 'ARID1A', 'RAC1', 'PPP6C', 'PREX2', 'IDH1', 'KIT', 'GNAQ', 'GNA11']


In [14]:
# -- Intersect tumour samples with MC3; zero-fill normals -----------------
# Normal samples (type 11) have no somatic mutations in MC3 by design.
# We keep them and assign all-zero mutation rows rather than dropping them,
# so the classifier sees genuine mutation-negative transcriptomes.
bulk_15   = pd.Index(adata_bulk.obs_names.str[:15])
mut_15    = pd.Index(mutation_labels.index.str[:15])
is_normal = adata_bulk.obs["sample_type"] == "normal"

has_mc3   = bulk_15.isin(mut_15)
keep_mask = has_mc3 | is_normal.values
print(f"Tumour samples with MC3 : {has_mc3.sum()}")
print(f"Normal samples kept     : {is_normal.sum()}")
print(f"Total retained          : {keep_mask.sum()} / {adata_bulk.n_obs}")
assert has_mc3.sum() > 50, "Suspiciously few MC3 matches -- check barcode format"

adata_bulk = adata_bulk[keep_mask].copy()
adata_bulk.obs["barcode_15"] = adata_bulk.obs_names.str[:15]

# Align mutation labels: tumour rows from MC3, normal rows = 0
mut_reindexed       = mutation_labels.copy()
mut_reindexed.index = mut_reindexed.index.str[:15]
mut_reindexed       = mut_reindexed[~mut_reindexed.index.duplicated(keep="first")]

# Reindex -- normals get NaN then fill 0
mutation_labels = mut_reindexed.reindex(adata_bulk.obs["barcode_15"]).fillna(0).astype(int)
mutation_labels.index = adata_bulk.obs_names

print(f"\nMutation matrix : {mutation_labels.shape}")
print("\nMutation frequencies (tumour samples only):")
tumour_mask = adata_bulk.obs["sample_type"] != "normal"
print(mutation_labels[tumour_mask].sum().sort_values(ascending=False))
mutation_labels.head()


Tumour samples with MC3 : 465
Normal samples kept     : 1
Total retained          : 466 / 473

Mutation matrix : (466, 26)

Mutation frequencies (tumour samples only):
BRAF      240
NRAS      128
PREX2      97
TP53       70
NF1        68
ARID2      61
CDKN2A     61
PTEN       44
APC        39
PPP6C      31
RAC1       29
MAP2K1     27
SETD2      26
CTNNB1     24
ARID1A     23
IDH1       23
KIT        21
GNA11      12
PIK3CA     11
KRAS       11
GNAQ       10
MAP2K2      9
AKT3        8
AKT1        6
HRAS        6
KDM6A       5
dtype: int64


,BRAF,NRAS,NF1,MAP2K1,MAP2K2,KRAS,HRAS,PTEN,PIK3CA,AKT1,...,SETD2,KDM6A,ARID1A,RAC1,PPP6C,PREX2,IDH1,KIT,GNAQ,GNA11
TCGA-YD-A89C-06,0,1,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
TCGA-Z2-AA3V-06,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
TCGA-EB-A3Y6-01,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
TCGA-EE-A3JA-06,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
TCGA-D9-A4Z2-01,1,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## 5. Load single-cell data

In [15]:
# -- Jerby-Arnon et al. 2018 GSE115978 -- log2(TPM+1) ----------------------
# File is genes x cells, already log2-scaled -- no CPM normalisation needed.
print("Loading TPM matrix ...")
tpm_df = pd.read_csv(sc_tpm_path, index_col=0)   # genes x cells
print(f"TPM raw : {tpm_df.shape}  (genes x cells)")


Loading TPM matrix ...
TPM raw : (23686, 7186)  (genes x cells)


In [16]:
# -- Cell metadata ---------------------------------------------------------
# GSE115978_cell.annotations.csv.gz is comma-separated, not TSV.
meta = pd.read_csv(sc_meta_path, index_col=0)
print(f"Metadata shape   : {meta.shape}")
print(f"Metadata columns : {meta.columns.tolist()}")
meta.head(3)


Metadata shape   : (7186, 6)
Metadata columns : ['samples', 'cell.types', 'treatment.group', 'Cohort', 'no.of.genes', 'no.of.reads']


,samples,cell.types,treatment.group,Cohort,no.of.genes,no.of.reads
cells,,,,,,
cy78_CD45_neg_1_B04_S496_comb,Mel78,Mal,post.treatment,Tirosh,8258,357919
cy79_p4_CD45_neg_PDL1_neg_E11_S1115_comb,Mel79,Mal,treatment.naive,Tirosh,2047,5727
CY88_5_B10_S694_comb,Mel88,Mal,post.treatment,Tirosh,5375,139218


In [17]:
# -- Build AnnData ---------------------------------------------------------
common_cells = tpm_df.columns.intersection(meta.index)
print(f"Cells in both matrix and metadata : {len(common_cells)}")

adata_sc = ad.AnnData(
    X   = tpm_df[common_cells].T.values.astype(np.float32),
    obs = meta.loc[common_cells],
    var = pd.DataFrame(index=tpm_df.index),
)
adata_sc.var_names_make_unique()
print(f"\nSC loaded : {adata_sc.n_obs} cells x {adata_sc.n_vars} genes")

ct_col = next(
    (c for c in adata_sc.obs.columns
     if "type" in c.lower() or "class" in c.lower()), None
)
if ct_col:
    print(f"\nCell type distribution ({ct_col}):")
    print(adata_sc.obs[ct_col].value_counts())


Cells in both matrix and metadata : 7186

SC loaded : 7186 cells x 23686 genes

Cell type distribution (cell.types):
cell.types
Mal           2018
T.CD8         1759
T.CD4          856
B.cell         818
T.cell         706
Macrophage     420
?              307
CAF            106
Endo.          104
NK              92
Name: count, dtype: int64


In [18]:
# -- QC (TPM already log2-scaled -- skip CPM normalisation) ----------------
sc.pp.filter_genes(adata_sc, min_cells=5)
sc.pp.filter_cells(adata_sc, min_genes=100)

print(f"After QC : {adata_sc.n_obs} cells x {adata_sc.n_vars} genes")

X_raw = adata_sc.X
print(f"Value range: {X_raw.min():.2f} - {X_raw.max():.2f}")
print("(GSE115978 TPM matrix is log2(TPM+1) -- no further normalisation applied.)")


After QC : 7186 cells x 22246 genes
Value range: 0.00 - 15.92
(GSE115978 TPM matrix is log2(TPM+1) -- no further normalisation applied.)


## 6. Gene overlap & sanity checks

In [19]:
overlap = set(adata_bulk.var_names) & set(adata_sc.var_names)
print(f"Bulk  : {adata_bulk.n_obs} samples x {adata_bulk.n_vars} genes")
print(f"SC    : {adata_sc.n_obs} cells x {adata_sc.n_vars} genes")
print(f"Muts  : {mutation_labels.shape}")
print(f"Gene overlap : {len(overlap):,}")

X = adata_bulk.X
print(f"\nBulk NaN : {np.isnan(X).sum()} | Inf : {np.isinf(X).sum()}")
print(f"\nMutation frequencies:")
print(mutation_labels.sum().sort_values(ascending=False))
mutation_labels.head()


Bulk  : 466 samples x 20530 genes
SC    : 7186 cells x 22246 genes
Muts  : (466, 26)
Gene overlap : 18,729

Bulk NaN : 0 | Inf : 0

Mutation frequencies:
BRAF      240
NRAS      128
PREX2      97
TP53       70
NF1        68
ARID2      61
CDKN2A     61
PTEN       44
APC        39
PPP6C      31
RAC1       29
MAP2K1     27
SETD2      26
CTNNB1     24
ARID1A     23
IDH1       23
KIT        21
GNA11      12
PIK3CA     11
KRAS       11
GNAQ       10
MAP2K2      9
AKT3        8
AKT1        6
HRAS        6
KDM6A       5
dtype: int64


,BRAF,NRAS,NF1,MAP2K1,MAP2K2,KRAS,HRAS,PTEN,PIK3CA,AKT1,...,SETD2,KDM6A,ARID1A,RAC1,PPP6C,PREX2,IDH1,KIT,GNAQ,GNA11
TCGA-YD-A89C-06,0,1,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
TCGA-Z2-AA3V-06,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
TCGA-EB-A3Y6-01,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
TCGA-EE-A3JA-06,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
TCGA-D9-A4Z2-01,1,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## 7. Phase 1 -- Bulk pipeline

Both bulk and SC data are log2-scaled, so `norm_method='none'` and
`log1p=False` are appropriate for both phases.


In [20]:
n_components = 120

bulk_pipe = BulkPipeline(
    norm_method         = "none",
    log1p               = False,
    center              = True,
    scale               = True,
    decomposition       = "svd",
    n_components        = n_components,
    classifier          = "logistic",
    min_positive_frac   = 0.0001,
    classifier_kwargs   = {
        "C"         : 1.0,
        "solver"    : "saga",     # might try "saga" solver vs "lbfgs"
        "max_iter"  : 1000000,
    },
)

bulk_pipe.fit(adata_bulk, mutation_labels, cv=10)
bulk_pipe.save(os.path.join(MODELS_DIR, "SKCM_bulk_pipeline.pkl"))
print("Phase 1 complete -- model saved.")


18:10:00 | INFO     | scope.pipeline.bulk_pipeline — === BulkPipeline.fit ===
18:10:00 | INFO     | scope.pipeline.bulk_pipeline — Preprocessing bulk data (norm=none, log1p=False).
18:10:00 | INFO     | scope.preprocessing.bulk — BulkNormalizer fitted (method=none).
18:10:00 | INFO     | scope.preprocessing.bulk — BulkScaler fitted (center=True, scale=True).
18:10:00 | INFO     | scope.preprocessing.bulk — BulkPreprocessor fitted: 20530 genes → 20530 after filtering.
18:10:00 | INFO     | scope.pipeline.bulk_pipeline — Decomposition: svd (k=120).
18:10:00 | INFO     | scope.decomposition.svd — SVD fitted: 120 components (cumulative EVR=0.751).
18:10:00 | INFO     | scope.pipeline.bulk_pipeline — Training classifiers (logistic).
/opt/homebrew/Cellar/micromamba/2.5.0_1/envs/scope-dev/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1221: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
18:10:00 | INFO     | scope.c

Phase 1 complete -- model saved.


In [21]:
scree = bulk_pipe.decomposer_.scree_data()
fig, ax = plot_scree(scree, max_components=n_components)
plt.tight_layout()
fig.savefig(os.path.join(MODELS_DIR, "SKCM_scree.pdf"), bbox_inches="tight")
plt.show()


/var/folders/fl/dl2g54g52svg861xm9w8vp6xtdpc9w/T/ipykernel_33180/1086793904.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Phase 2 -- Single-cell pipeline

In [22]:
adata_bulk_pp = bulk_pipe.preprocessor_.transform(adata_bulk)

# SingleCellPipeline internally calls log1p on whatever it receives.
# Our data is already log2(TPM+1), so we de-log back to raw TPM first,
# then clamp negatives (a known artefact in the Jerby-Arnon matrix).
adata_sc_pipe = adata_sc.copy()
adata_sc_pipe.X = np.clip((2 ** adata_sc.X) - 1, 0, None)

sc_pipe = SingleCellPipeline(
    bulk_pipeline      = bulk_pipe,
    alignment_method   = "moment_matching",
    sc_filter_strategy = "none",   # QC already applied upstream
)
sc_pipe.fit(adata_bulk_pp, adata_sc_pipe)
adata_sc = sc_pipe.transform(adata_sc_pipe)

mut_prob_cols = [c for c in adata_sc.obs.columns if c.startswith("mutation_prob_")]
print(f"Inferred {len(mut_prob_cols)} mutation probability columns")


18:12:11 | INFO     | scope.pipeline.sc_pipeline — === SingleCellPipeline.fit ===
18:12:11 | INFO     | scope.preprocessing.single_cell — SingleCellPreprocessor fitted.
18:12:12 | INFO     | scope.utils.gene_utils — Gene universe: 18729 shared, 1801 bulk-only, 3517 sc-only
18:12:14 | INFO     | scope.preprocessing.alignment — BulkSCAligner fitted (method=moment_matching, n_genes=18729).
18:12:14 | INFO     | scope.pipeline.sc_pipeline — SingleCellPipeline.fit complete.
18:12:14 | INFO     | scope.pipeline.sc_pipeline — === SingleCellPipeline.transform ===
18:12:33 | WARNING  | scope.pipeline.sc_pipeline — 1801 / 20530 bulk genes absent from sc data (will be zero-padded).
18:12:45 | INFO     | scope.pipeline.sc_pipeline — Projected 7186 cells into 120-D latent space.
18:12:45 | INFO     | scope.pipeline.sc_pipeline — Inferred probabilities for 26 mutations.


Inferred 26 mutation probability columns


## 9. Post-hoc probability calibration

The logistic classifier outputs probabilities relative to the bulk training
distribution. With ~50% BRAF-mutant samples, the prior is high and TME cells
inherit an elevated floor. We apply **isotonic regression calibration** fitted
on bulk leave-one-out predictions vs. known mutation labels, then re-apply to
the SC probabilities. This preserves rank order while correcting the baseline.


In [23]:
from sklearn.isotonic import IsotonicRegression

adata_bulk_emb = bulk_pipe.decomposer_.transform(adata_bulk_pp)
Z_bulk         = adata_bulk_emb.obsm[bulk_pipe.obsm_key_]
bulk_probs_df  = bulk_pipe.classifier_set_.predict_proba(Z_bulk)

calibrators = {}
for col in bulk_probs_df.columns:                        # col = 'mutation_prob_BRAF'
    gene = col.replace("mutation_prob_", "")             # gene = 'BRAF'
    if gene not in mutation_labels.columns:
        continue
    y_true  = mutation_labels[gene].values
    y_score = bulk_probs_df[col].values
    if y_true.sum() < 10:
        continue
    iso = IsotonicRegression(out_of_bounds="clip")
    iso.fit(y_score, y_true)
    calibrators[gene] = iso

print(f"Fitted calibrators for {len(calibrators)} / {len(bulk_probs_df.columns)} genes")

for gene, iso in calibrators.items():
    raw_col   = f"mutation_prob_{gene}"
    calib_col = f"mutation_prob_cal_{gene}"
    if raw_col in adata_sc.obs.columns:
        adata_sc.obs[calib_col] = iso.predict(adata_sc.obs[raw_col].values)

calib_cols = [c for c in adata_sc.obs.columns if c.startswith("mutation_prob_cal_")]
print(f"Calibrated columns written: {calib_cols}")


Fitted calibrators for 21 / 26 genes
Calibrated columns written: ['mutation_prob_cal_BRAF', 'mutation_prob_cal_NRAS', 'mutation_prob_cal_NF1', 'mutation_prob_cal_MAP2K1', 'mutation_prob_cal_KRAS', 'mutation_prob_cal_PTEN', 'mutation_prob_cal_PIK3CA', 'mutation_prob_cal_TP53', 'mutation_prob_cal_CDKN2A', 'mutation_prob_cal_CTNNB1', 'mutation_prob_cal_APC', 'mutation_prob_cal_ARID2', 'mutation_prob_cal_SETD2', 'mutation_prob_cal_ARID1A', 'mutation_prob_cal_RAC1', 'mutation_prob_cal_PPP6C', 'mutation_prob_cal_PREX2', 'mutation_prob_cal_IDH1', 'mutation_prob_cal_KIT', 'mutation_prob_cal_GNAQ', 'mutation_prob_cal_GNA11']


In [24]:
# -- Diagnostic: P(BRAF) distribution by cell type, raw vs calibrated -----
ct_col = next((c for c in adata_sc.obs.columns
               if "type" in c.lower() or "class" in c.lower()), None)

if ct_col and "mutation_prob_cal_BRAF" in adata_sc.obs.columns:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=False)
    cell_types = adata_sc.obs[ct_col].unique()

    for ax, col, title in zip(
        axes,
        ["mutation_prob_BRAF", "mutation_prob_cal_BRAF"],
        ["P(BRAF) -- raw", "P(BRAF) -- isotonic calibrated"],
    ):
        data   = [adata_sc.obs.loc[adata_sc.obs[ct_col] == ct, col].values
                  for ct in cell_types]
        vp = ax.violinplot(data, showmedians=True)
        ax.set_xticks(range(1, len(cell_types) + 1))
        ax.set_xticklabels(cell_types, rotation=45, ha="right")
        ax.set_ylabel("Probability")
        ax.set_title(title)
        ax.axhline(0.5, color="grey", lw=0.8, linestyle="--", label="0.5")
    plt.tight_layout()
    fig.savefig(os.path.join(MODELS_DIR, "SKCM_BRAF_calibration_diagnostic.pdf"),
                bbox_inches="tight")
    plt.show()
else:
    print("ct_col or calibrated BRAF column not found -- skipping diagnostic.")


/var/folders/fl/dl2g54g52svg861xm9w8vp6xtdpc9w/T/ipykernel_33180/1723238369.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 10. Visualisation


In [25]:
adata_sc = compute_umap(adata_sc, obsm_key="X_svd")

ct_col = next((c for c in adata_sc.obs.columns
               if "type" in c.lower() or "class" in c.lower()), "sample_type")

# Prefer calibrated BRAF prob if available
#braf_col = ("mutation_prob_cal_BRAF"
#            if "mutation_prob_cal_BRAF" in adata_sc.obs.columns
#            else "mutation_prob_BRAF")
braf_col = ("mutation_prob_BRAF")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sc.pl.umap(adata_sc, color=ct_col,   ax=axes[0], show=False, title="Cell type")
sc.pl.umap(adata_sc, color=braf_col, ax=axes[1], show=False,
           title=f"P(BRAF mutated) [{braf_col}]")
plt.tight_layout()
fig.savefig(os.path.join(MODELS_DIR, "SKCM_umap_BRAF.pdf"), bbox_inches="tight")
plt.show()


/opt/homebrew/Cellar/micromamba/2.5.0_1/envs/scope-dev/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
18:13:15 | INFO     | scope.visualization.embeddings — UMAP computed: 7186 cells → 2D.
/var/folders/fl/dl2g54g52svg861xm9w8vp6xtdpc9w/T/ipykernel_33180/3577527357.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [26]:
from sklearn.pipeline import Pipeline
import numpy as np
import pandas as pd

macrophage_markers = ['CD68', 'CD163', 'CSF1R', 'MRC1', 'C1QA', 'APOE', 'LYZ']
gene_names = bulk_pipe.gene_names_
Vt = bulk_pipe.decomposer_.components_  # shape: (n_components, n_genes)

for gene in ['BRAF', 'NRAS', 'NF1']:
    if gene not in bulk_pipe.classifier_set_.classifiers_:
        print(f"\n=== {gene} — not in classifier set, skipping ===")
        continue

    clf = bulk_pipe.classifier_set_.classifiers_[gene]
    inner = clf._clf
    coefs = inner[-1].coef_[0] if isinstance(inner, Pipeline) else inner.coef_[0]

    weighted = (Vt * coefs[:, None]).sum(axis=0)  # (n_genes,)
    weighted_series = pd.Series(weighted, index=gene_names).abs().sort_values(ascending=False)

    print(f"\n=== {gene} ===")
    print("Top 20 genes:")
    print(weighted_series.head(20))

    for m in macrophage_markers:
        if m in weighted_series.index:
            rank = weighted_series.index.tolist().index(m) + 1
            print(f"  {m} rank: {rank} / {len(gene_names)}")
        else:
            print(f"  {m} not in gene set")
            


=== BRAF ===
Top 20 genes:
WDR91       0.125943
MLEC        0.119852
CNOT4       0.118124
SUSD5       0.117789
WBSCR22     0.114558
ZNF540      0.113477
VDAC2       0.112683
SLC22A10    0.111920
PPRC1       0.111738
SNX10       0.109943
FAM84B      0.109008
ZC3HC1      0.108767
SSBP1       0.108163
PON2        0.106352
CNPY4       0.105996
TSGA14      0.105898
FKBP9L      0.105345
DNAJB6      0.105006
CD200       0.104956
MRPS33      0.104659
dtype: float64
  CD68 rank: 18459 / 20530
  CD163 rank: 19939 / 20530
  CSF1R rank: 20222 / 20530
  MRC1 rank: 10431 / 20530
  C1QA rank: 18262 / 20530
  APOE rank: 16911 / 20530
  LYZ rank: 4987 / 20530

=== NRAS ===
Top 20 genes:
C7orf42         0.207925
RABGEF1         0.188471
NDUFB2          0.183741
PSMC2           0.181092
CRCP            0.174580
FAM40A          0.174472
NRAS            0.173567
BCL7B           0.172254
PMS2L1          0.168507
WDR77           0.164591
SYPL1           0.162264
SRPK2           0.162092
DCLRE1B         0.16

In [27]:
top_muts = (
    mutation_labels.sum().sort_values(ascending=False).index.tolist()
)
fig = plot_mutation_probabilities(adata_sc, mutations=top_muts)
fig.savefig(os.path.join(MODELS_DIR, "SKCM_mutation_probs.pdf"), bbox_inches="tight")
plt.show()


/var/folders/fl/dl2g54g52svg861xm9w8vp6xtdpc9w/T/ipykernel_33180/1893339959.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [28]:
top_muts = mutation_labels.sum().sort_values(ascending=False).index.tolist()

prob_cols = [f"mutation_prob_{g}" for g in top_muts if f"mutation_prob_{g}" in adata_sc.obs.columns]

# Per-gene vmax at 99th percentile -- keeps each panel's colorbar meaningful
vmaxes = {c: max(float(np.percentile(adata_sc.obs[c].values, 99)), 0.0005) for c in prob_cols}

ncols = 3
nrows = int(np.ceil(len(prob_cols) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5 * nrows))
axes = axes.flatten()

for i, col in enumerate(prob_cols):
    gene = col.replace("mutation_prob_", "")
    sc.pl.umap(
        adata_sc,
        color=col,
        ax=axes[i],
        show=False,
        title=f"P({gene} mut)",
        vmin=0,
        vmax=vmaxes[col],
        cmap="RdBu_r",
        colorbar_loc="right",
        s=8,
    )

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
fig.savefig(os.path.join(MODELS_DIR, "SKCM_mutation_probs.pdf"), bbox_inches="tight")
plt.show()


/var/folders/fl/dl2g54g52svg861xm9w8vp6xtdpc9w/T/ipykernel_33180/3032322867.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [29]:
fig, ax = plot_mutation_heatmap(adata_sc, cluster_key=ct_col, mutations=top_muts)

# Resize after the fact
fig.set_size_inches(14, 8)  # (width, height) in inches

fig.savefig(
    os.path.join(MODELS_DIR, "SKCM_heatmap.pdf"),
    bbox_inches="tight",
    dpi=300        # for raster formats (PNG, etc.)
)


/opt/homebrew/Cellar/micromamba/2.5.0_1/envs/scope-dev/lib/python3.10/site-packages/scope/visualization/embeddings.py:417: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  mean_prob = df.groupby(cluster_key)[prob_cols].mean()


In [30]:
import seaborn as sns
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests

def plot_mutation_cooccurrence(mutation_labels, title="Mutation Co-occurrence", 
                                min_freq=5, method="jaccard", figsize=(12, 10)):
    """
    Parameters
    ----------
    mutation_labels : pd.DataFrame
        Binary matrix (samples x genes)
    min_freq : int
        Minimum mutation count to include gene
    method : str
        'jaccard'  — co-occurrence as Jaccard index (size-normalized)
        'phi'      — phi coefficient (correlation on binary vars)
        'log_odds' — log odds ratio from Fisher's exact test
    """
    # Filter low-frequency mutations
    freq = mutation_labels.sum()
    mat  = mutation_labels.loc[:, freq >= min_freq].astype(int)
    genes = mat.columns.tolist()
    n     = len(genes)
    
    # Build co-occurrence matrix
    cooc  = np.zeros((n, n))
    pvals = np.ones((n, n))
    
    for i, g1 in enumerate(genes):
        for j, g2 in enumerate(genes):
            a = mat[g1].values
            b = mat[g2].values
            
            if method == "jaccard":
                intersection = (a & b).sum()
                union        = (a | b).sum()
                cooc[i, j]   = intersection / union if union > 0 else 0
                
            elif method == "phi":
                # Phi coefficient = Pearson on binary
                cooc[i, j] = np.corrcoef(a, b)[0, 1]
                
            elif method == "log_odds":
                # 2x2 contingency
                tp = (a & b).sum()
                fp = (~a.astype(bool) & b.astype(bool)).sum()
                fn = (a.astype(bool) & ~b.astype(bool)).sum()
                tn = (~a.astype(bool) & ~b.astype(bool)).sum()
                table = [[tp, fp], [fn, tn]]
                _, p  = fisher_exact(table)
                pvals[i, j] = p
                or_val = (tp * tn) / (fp * fn) if (fp * fn) > 0 else np.nan
                cooc[i, j] = np.log2(or_val) if or_val and or_val > 0 else 0

    cooc_df = pd.DataFrame(cooc, index=genes, columns=genes)
    
    # FDR correction for log_odds
    annot_df = None
    if method == "log_odds":
        flat_p = pvals[np.triu_indices(n, k=1)]
        _, qvals, _, _ = multipletests(flat_p, method='fdr_bh')
        sig = np.ones((n, n), dtype=bool)
        idx = np.triu_indices(n, k=1)
        sig[idx] = qvals < 0.05
        sig = sig | sig.T
        annot_df = pd.DataFrame(
            np.where(sig, "", "ns"), index=genes, columns=genes
        )
    
    # Sort by mutation frequency descending
    order    = mat.sum().sort_values(ascending=False).index
    cooc_df  = cooc_df.loc[order, order]
    freq_sorted = mat.sum()[order]
    
    # Plot
    fig, ax = plt.subplots(figsize=figsize)
    
    cmap   = "RdBu_r" if method in ("phi", "log_odds") else "YlOrRd"
    center = 0         if method in ("phi", "log_odds") else None
    
    sns.heatmap(
        cooc_df,
        ax        = ax,
        cmap      = cmap,
        center    = center,
        square    = True,
        linewidths= 0.5,
        linecolor = "white",
        annot     = annot_df.loc[order, order] if annot_df is not None else False,
        fmt       = "",
        cbar_kws  = {"shrink": 0.8, "label": method},
    )
    
    # Annotate diagonal with frequencies
    for k, gene in enumerate(order):
        ax.text(k + 0.5, k + 0.5, str(int(freq_sorted[gene])),
                ha="center", va="center", fontsize=7,
                color="white", fontweight="bold")
    
    ax.set_title(f"{title}\n(diagonal = mutation count, min_freq ≥ {min_freq})",
                 fontsize=13, pad=12)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=9)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=9)
    plt.tight_layout()
    return fig


# --- SKCM ---
fig_skcm = plot_mutation_cooccurrence(
    mutation_labels,
    title   = "SKCM Mutation Co-occurrence",
    min_freq= 5,
    method  = "phi",       # swap to "jaccard" or "log_odds" as needed
)
fig_skcm.savefig(os.path.join(MODELS_DIR, "SKCM_mutation_cooccurrence.pdf"), bbox_inches="tight")
plt.show()


/var/folders/fl/dl2g54g52svg861xm9w8vp6xtdpc9w/T/ipykernel_33180/2548332994.py:116: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [31]:
def plot_mutation_clustermap(
    gene,
    Vt,
    gene_names,
    bulk_pipe,
    adata_bulk,
    adata_sc,
    mutation_labels,
    n_genes=50,
    figsize=(18, 12),
    sc_ct_col="cell_type",
):
    from scipy.stats import zscore
    from matplotlib.patches import Patch

    # ── 1. Weighted gene loadings ─────────────────────────────────────────
    clf      = bulk_pipe.classifier_set_.classifiers_[gene]
    coefs    = clf._clf.coef_[0]
    weighted = pd.Series((Vt * coefs[:, None]).sum(axis=0), index=gene_names)

    top_genes = weighted.sort_values(ascending=False).head(n_genes).index
    bot_genes = weighted.sort_values(ascending=True ).head(n_genes).index
    sel_genes = top_genes.tolist() + bot_genes.tolist()

    # ── 2. Pull expression matrices ───────────────────────────────────────
    bulk_genes = adata_bulk.var_names.tolist()
    sc_genes   = adata_sc.var_names.tolist()
    sel_shared = [g for g in sel_genes if g in bulk_genes and g in sc_genes]

    X_bulk = pd.DataFrame(
        adata_bulk[:, sel_shared].X,
        index=adata_bulk.obs_names, columns=sel_shared,
    )
    X_sc = pd.DataFrame(
        adata_sc[:, sel_shared].X,
        index=adata_sc.obs_names, columns=sel_shared,
    )

    # ── 3. Z-score, drop near-constant genes, fill NaNs ──────────────────
    Z_bulk = pd.DataFrame(
        zscore(X_bulk, axis=0), index=X_bulk.index, columns=X_bulk.columns
    ).clip(-3, 3)
    Z_sc = pd.DataFrame(
        zscore(X_sc, axis=0), index=X_sc.index, columns=X_sc.columns
    ).clip(-3, 3)

    valid_genes = (
        Z_bulk.columns[Z_bulk.notna().all(axis=0)]
        .intersection(Z_sc.columns[Z_sc.notna().all(axis=0)])
    )
    Z_bulk = Z_bulk[valid_genes].fillna(0)
    Z_sc   = Z_sc[valid_genes].fillna(0)

    print(f"[{gene}] {len(sel_shared)} selected genes → {len(valid_genes)} after dropping near-constant")

    bulk_sample = Z_bulk.sample(min(200, len(Z_bulk)), random_state=42)
    sc_sample   = Z_sc.sample(  min(300, len(Z_sc)),   random_state=42)
    Z_combined  = pd.concat([bulk_sample, sc_sample], axis=0)

    # ── 4. Row colour bars ────────────────────────────────────────────────
    source_labels = ["Bulk"] * len(bulk_sample) + ["SC"] * len(sc_sample)
    source_pal    = {"Bulk": "#2C7BB6", "SC": "#D7191C"}
    row_source    = pd.Series(source_labels, index=Z_combined.index).map(source_pal)

    mut_status = []
    for idx in Z_combined.index:
        if idx in bulk_sample.index:
            barcode = idx[:15]
            if barcode in mutation_labels.index and gene in mutation_labels.columns:
                mut_status.append(int(mutation_labels.loc[barcode, gene]))
            else:
                mut_status.append(np.nan)
        else:
            mut_status.append(np.nan)

    mut_pal = {1: "#B2182B", 0: "#4393C3"}
    row_mut = pd.Series(mut_status, index=Z_combined.index).map(
        lambda x: mut_pal.get(x, "#CCCCCC")
    )

    if sc_ct_col in adata_sc.obs.columns:
        ct_vals = adata_sc.obs[sc_ct_col]
        ct_cats = ct_vals.unique()
        ct_pal  = dict(zip(ct_cats, sns.color_palette("tab20", len(ct_cats))))
        row_ct  = pd.Series(
            [ct_pal.get(ct_vals.get(i, None), "#CCCCCC") for i in Z_combined.index],
            index=Z_combined.index,
        )
        row_colors = pd.DataFrame({
            "Source":     row_source,
            "Mut status": row_mut,
            "Cell type":  row_ct,
        })
    else:
        row_colors = pd.DataFrame({
            "Source":     row_source,
            "Mut status": row_mut,
        })

    # ── 5. Column colour bar ──────────────────────────────────────────────
    color_map = {g: "#B2182B" for g in top_genes if g in valid_genes}
    color_map.update({g: "#2166AC" for g in bot_genes if g in valid_genes})
    col_colors = pd.Series(
        [color_map[g] for g in valid_genes],
        index=valid_genes,
        name="Loading direction",
    )

    # ── 6. Clustermap ─────────────────────────────────────────────────────
    g = sns.clustermap(
        Z_combined,
        row_colors      = row_colors,
        col_colors      = col_colors,
        cmap            = "RdBu_r",
        center          = 0,
        vmin=-3, vmax=3,
        figsize         = figsize,
        xticklabels     = True,
        yticklabels     = False,
        row_cluster     = True,
        col_cluster     = True,
        dendrogram_ratio= (0.1, 0.02),
        colors_ratio    = 0.02,
        cbar_pos        = (0.02, 0.82, 0.03, 0.08),
        linewidths      = 0,
    )
    g.ax_heatmap.set_xticklabels(
        g.ax_heatmap.get_xticklabels(), rotation=90, fontsize=6
    )

    # ── 7. Fix top gap by manually repositioning axes ─────────────────────
    heatmap_pos   = g.ax_heatmap.get_position()
    col_color_pos = g.ax_col_colors.get_position()
    col_dend_pos  = g.ax_col_dendrogram.get_position()

    gap         = 0.005
    dend_height = 0.03

    # Pin color bar flush to top of heatmap
    g.ax_col_colors.set_position([
        col_color_pos.x0,
        heatmap_pos.y1 + gap,
        col_color_pos.width,
        col_color_pos.height,
    ])
    # Pin dendrogram flush above color bar
    g.ax_col_dendrogram.set_position([
        col_dend_pos.x0,
        heatmap_pos.y1 + col_color_pos.height + gap * 2,
        col_dend_pos.width,
        dend_height,
    ])

    g.fig.suptitle(
        f"{gene} — top/bottom {n_genes} predictive genes\n"
        f"(red col = positive loading, blue col = negative loading)",
        y=0.95, fontsize=16,
    )
    g.fig.subplots_adjust(top=0.90)

    # ── 8. Legend ─────────────────────────────────────────────────────────
    legend_elements = [
        Patch(facecolor="#2C7BB6", label="Bulk"),
        Patch(facecolor="#D7191C", label="SC"),
        Patch(facecolor="#B2182B", label=f"{gene} mutated"),
        Patch(facecolor="#4393C3", label=f"{gene} WT"),
        Patch(facecolor="#CCCCCC", label="Unknown / SC"),
    ]
    g.ax_heatmap.legend(
        handles=legend_elements, bbox_to_anchor=(1.25, 1.1),
        loc="upper left", frameon=True, fontsize=8,
    )
    return g


In [32]:
# ── Run ───────────────────────────────────────────────────────────────────
for gene in ['BRAF', 'NRAS', 'NF1']:
    g = plot_mutation_clustermap(
        gene=gene, Vt=Vt, gene_names=gene_names,
        bulk_pipe=bulk_pipe, adata_bulk=adata_bulk,
        adata_sc=adata_sc, mutation_labels=mutation_labels,
        n_genes=50, sc_ct_col="cell.type",
    )
    g.fig.savefig(
        os.path.join(MODELS_DIR, f"SKCM_{gene}_clustermap.pdf"),
        bbox_inches="tight", dpi=150,
    )
    plt.show()
    

[BRAF] 100 selected genes → 92 after dropping near-constant


/var/folders/fl/dl2g54g52svg861xm9w8vp6xtdpc9w/T/ipykernel_33180/783852608.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/fl/dl2g54g52svg861xm9w8vp6xtdpc9w/T/ipykernel_33180/2684711861.py:44: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  zscore(X_sc, axis=0), index=X_sc.index, columns=X_sc.columns


[NRAS] 100 selected genes → 86 after dropping near-constant


/var/folders/fl/dl2g54g52svg861xm9w8vp6xtdpc9w/T/ipykernel_33180/783852608.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


[NF1] 100 selected genes → 93 after dropping near-constant


/var/folders/fl/dl2g54g52svg861xm9w8vp6xtdpc9w/T/ipykernel_33180/783852608.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [33]:
import anndata as ad
import pandas as pd
from scope import BulkPipeline
from scope.io import load_mutation_labels
from scope.evaluation import SVDEvaluator

# ── Get Z_bulk ──────────────────────────────────────────────────────────────
adata_pp = bulk_pipe.transform_bulk(adata_bulk)
Z_bulk = adata_pp.obsm[bulk_pipe.obsm_key_]

# ── Run SVDEvaluator for each mutation ──────────────────────────────────────
mutations = list(bulk_pipe.classifier_set_.classifiers_.keys())
print(f"Mutations to evaluate: {mutations}")

for mutation in mutations:
    print(f"\n── Evaluating {mutation} ──")
    ev = SVDEvaluator(bulk_pipe, Z_bulk, mutation=mutation)
    ev.run_all(
        output_dir=f"figures/svd_eval_{mutation}",
        cv=5,
        n_permutations=200,
    )
    print(f"  ✓ Saved to figures/svd_eval_{mutation}/")
    

18:13:34 | INFO     | scope.evaluation.svd_evaluation — === SVDEvaluator running for mutation 'BRAF' → figures/svd_eval_BRAF ===
18:13:34 | INFO     | scope.evaluation.svd_evaluation —   saved → figures/svd_eval_BRAF/weighted_scree.png


Mutations to evaluate: ['BRAF', 'NRAS', 'NF1', 'MAP2K1', 'MAP2K2', 'KRAS', 'HRAS', 'PTEN', 'PIK3CA', 'AKT1', 'AKT3', 'TP53', 'CDKN2A', 'CTNNB1', 'APC', 'ARID2', 'SETD2', 'KDM6A', 'ARID1A', 'RAC1', 'PPP6C', 'PREX2', 'IDH1', 'KIT', 'GNAQ', 'GNA11']

── Evaluating BRAF ──


18:13:34 | INFO     | scope.evaluation.svd_evaluation —   saved → figures/svd_eval_BRAF/component_importance.png
18:13:34 | INFO     | scope.evaluation.svd_evaluation —   saved → figures/svd_eval_BRAF/gene_loading_heatmap.png
18:13:35 | INFO     | scope.evaluation.svd_evaluation —   saved → figures/svd_eval_BRAF/top_genes_per_component.png
18:13:35 | INFO     | scope.evaluation.svd_evaluation —   saved → figures/svd_eval_BRAF/latent_scatter.png
18:13:36 | INFO     | scope.evaluation.svd_evaluation —   saved → figures/svd_eval_BRAF/separation_violins.png
18:13:36 | INFO     | scope.evaluation.svd_evaluation —   saved → figures/svd_eval_BRAF/component_label_correlation.png
/opt/homebrew/Cellar/micromamba/2.5.0_1/envs/scope-dev/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=

  ✓ Saved to figures/svd_eval_BRAF/

── Evaluating NRAS ──


18:13:56 | INFO     | scope.evaluation.svd_evaluation —   saved → figures/svd_eval_NRAS/component_importance.png
18:13:57 | INFO     | scope.evaluation.svd_evaluation —   saved → figures/svd_eval_NRAS/gene_loading_heatmap.png
18:13:57 | INFO     | scope.evaluation.svd_evaluation —   saved → figures/svd_eval_NRAS/top_genes_per_component.png
18:13:57 | INFO     | scope.evaluation.svd_evaluation —   saved → figures/svd_eval_NRAS/latent_scatter.png
18:13:58 | INFO     | scope.evaluation.svd_evaluation —   saved → figures/svd_eval_NRAS/separation_violins.png
18:13:58 | INFO     | scope.evaluation.svd_evaluation —   saved → figures/svd_eval_NRAS/component_label_correlation.png
/opt/homebrew/Cellar/micromamba/2.5.0_1/envs/scope-dev/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=